# EDA

1. Dataset Overview
2. Data Quality Overview   
3. Exploratory Analysis
4. BNF Chapters
5. Product-Level Analysis
6. Geographic Analysis
7. ICB Analysis
8. Temporal Analysis
9. EDA Summary


In [ ]:
# importing necessary files

import pandas as pd
import numpy as np

# importing plotting functions library
import sys
sys.path.append("../src")
import Plotting_Functions as pf

# setting for float display
pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.max_columns', None)  # ensuring all columns are visible

In [ ]:
# creating output folders to save data summaries generated during analysis

from pathlib import Path

# getting the output directory
Output_dir = Path("../output")

# setting up the folder names
folders = [
    Output_dir / "BNF Chapter",
    Output_dir / "Product_Medicine",
    Output_dir / "Geography",
    Output_dir / "ICB",
    Output_dir / "Temporal"
]

# creating the folders in a for loop
for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)


In [ ]:
# loading the cleanded dataset

df_NHS_dataset = pd.read_parquet("../data/cleaned data/nhs_prescribing_2026_jan_apr.parquet")

df_NHS_dataset.head(2)

In [ ]:
# checking the datatype for snomed_codes
df_NHS_dataset.info()

## SECTION 1 : Dataset Overview

Dataset Overview

    1.1 Dataset Dimensions
    1.2 Data Structure
    1.3 Time Period Coverage
    1.4 Key Entities and Measures

In [ ]:
# dataset dimensions

rows, columns = df_NHS_dataset.shape

print (f"Number of rows in this dataset: {rows}")
print (f"Number of columns in this dataset: {columns}")

The final analytical dataset contains 2.25 million records across 32 columns. Each record represents monthly prescribing activity for a medicine presentation, including prescribing volume, expenditure, geographical attributes, and therapeutic classifications within the BNF hierarchy.

In [ ]:
# data structure
df_NHS_dataset.info()

In [ ]:
df_NHS_dataset.select_dtypes(include=["number"]).columns

In [ ]:
df_NHS_dataset.select_dtypes(include=["bool", "datetime"]).columns

In [ ]:
df_NHS_dataset.select_dtypes(include=["string", "str"]).columns

The dataset contains a combination of numerical, categorical, temporal, and boolean variables. Numerical variables represent prescribing activity and expenditure measures, including items, total quantity dispensed, and NIC. Categorical variables capture medicine characteristics, geographical attributes, and therapeutic classifications within the BNF hierarchy. A dedicated date field and time-based variables support monthly trend analysis, while the exception record flag identifies records requiring separate consideration during analysis.

In [ ]:
# time period coverage

df_NHS_dataset["date"].min()  # checking the min date in the dataset (Jan 2026)

df_NHS_dataset["date"].max()  # checking the max date in the dataset (Apr 2026)

# the dataset has data covering the time period from Jan - Apr 2026


In [ ]:
# checking the unique months
#df_NHS_dataset["month_name"].unique()

df_NHS_dataset["date"].drop_duplicates().dt.strftime("%B %Y")

The dataset covers four consecutive months of prescribing activity, from January 2026 to April 2026. This enables comparisons of prescribing volume and expenditure over time, while providing a consistent quarterly view of prescribing patterns.

##### key entities and measures

Key Entities
- Medicine presentation
- Generic medicine
- Chemical substance
- BNF therapeutic classifications (Chapter, Section, Paragraph)
- Region
- Integrated Care Board (ICB)
- Supplier
- Time (Month)
  
Key Measures
- Items : Number of prescription items
- Total Quantity : Quantity of medicine dispensed
- NIC : Net Ingredient Cost of prescribed medicines

## SECTION 2 : Data Quality Overview

Data Quality Overview

    2.1 Missing Values
    2.2 Duplicate Records
    2.3 Numerical Validation
    2.4 Exception Records

In [ ]:
# missing values

df_NHS_dataset.isnull().sum().sort_values(ascending=False)
# only three columns have missing values.  The column with the highest nulls is the supplier name, followed by snomed_code.

In [ ]:
# showing missing values as a bar plot
missing = df_NHS_dataset.isnull().sum()

# creating a bar chart for missing values using the plotting function library
pf.plot_bar(df = missing[missing > 0].sort_values(ascending=False), 
            title = "Missing values by column", 
            ylabel = "Number of Missing values") 

In [ ]:
# missing values by percentage
missing_pct = (
    df_NHS_dataset.isnull().mean() * 100
).sort_values(ascending=False)

missing_pct = missing_pct[missing_pct > 0]

missing_pct

In [ ]:
# duplicated records

df_NHS_dataset.duplicated().sum()
# no duplicated values found

In [ ]:
# numerical validation

df_NHS_dataset[["items", "total_quantity", "nic"]].describe() # no negative values, within reasonable range

In [ ]:
# exception records

exception_count = df_NHS_dataset["is_exception_record"].sum()
exception_percentage = round((exception_count / len(df_NHS_dataset)) * 100, 2)

print (f"Number of exception records are: {exception_count}")
print (f"Exception records percentage is: {exception_percentage}%")

Data quality checks confirmed that the dataset was suitable for exploratory analysis. No duplicate records or negative values were identified within numerical measures. Missing values were minimal, with the majority of records containing complete prescribing information. Exception records were retained and flagged to maintain transparency and allow appropriate handling during analysis.

## SECTION 3 : Exploratory Analysis

Exploratory Analysis Overview

    3.1 Numerical Measures Overview
    3.2 Skewness check
    3.3 Visualisations

In [ ]:
# numerical measures overview

# look at the numerical measures
df_NHS_dataset[["items", "total_quantity", "nic"]].describe()

In [ ]:
# check skewness
df_NHS_dataset[["items", "total_quantity", "nic"]].skew() # positive values means heavily right skewed

#### VISUALISATIONS

In [ ]:
# boxplot of nic (log scale)

# calling boxplot function

pf.plot_boxplot(
        vals = df_NHS_dataset["nic"],
        plotLog = True,
        title="Distribution of Net Ingredient Cost",
        xlabel="",
        ylabel="NIC"
)

In [ ]:
# histogram of nic using log scale

# calling the plot histogram function
pf.plot_histogram(
    vals = df_NHS_dataset["nic"],
    plotLog= True,
    title= "Distribution of Net Ingredient Cost",
    ylabel = "Frequency",
    xlabel= "NIC"

)

In [ ]:
# histogram of items (log scale)

# calling the plot histogram function
pf.plot_histogram(
vals = df_NHS_dataset["items"],
    plotLog= True,
    title= "Distribution of Prescription Items",
    ylabel = "Frequency",
    xlabel= "Items"
)


In [ ]:
# histogram of total quantity (log scale)

pf.plot_histogram(
    vals = df_NHS_dataset["total_quantity"],
    plotLog= True,
    title= "Distribution of Total Quantity",
    ylabel = "Frequency",
    xlabel= "Quantity"

)

Distribution analysis showed that prescribing volume (items), quantity dispensed, and expenditure (NIC) were all heavily right-skewed. The majority of prescribing records had relatively low values, while a small number of records had substantially higher values, creating long right-hand tails in the distributions. Log-transformed visualisations were used to improve interpretation of these measures and better visualise patterns across the dataset.

## SECTION 4 : BNF Chapters

BNF Chapters Overview

    4.1 BNF Chapter Names
    4.2 Prescribing volume by Chapters
    4.3 Top 10 BNF Chapters by Net Ingredient Cost (NIC)
    4.4 Top 10 BNF Chapters by prescription Items
    4.5 Top 10 BNF chapters by total quatity dispensed 

In [ ]:
# naming the BNF chapters 

df_NHS_dataset["bnf_chapter"].value_counts().sort_index() 

In [ ]:
# prescribing volume by chapters

chapter_summary = (
                    df_NHS_dataset.groupby("bnf_chapter")[["items", "total_quantity", "nic"]].sum().sort_values("nic", ascending=False).reset_index()

)
chapter_summary

Aggregation by BNF chapter revealed substantial variation in prescribing activity and expenditure across therapeutic areas. Cardiovascular System accounted for the highest prescribing volume, while Endocrine System generated the greatest total expenditure despite a lower prescribing volume. Nutrition and Blood recorded the largest quantity dispensed, whereas chapters such as Appliances and Malignant Disease and Immunosuppression demonstrated comparatively high expenditure relative to prescribing volume. These findings suggest that prescribing frequency and cost are not directly proportional and vary considerably between therapeutic categories.

In [ ]:
# Top 10 BNF Chapters by Net Ingredient Cost (NIC)

top_chapter_nic = chapter_summary.nlargest(10,"nic").sort_values("nic", ascending=False)

# creating a bar chart for top 10 BNF chapters by NIC using the plotting function library
pf.plot_barh(df = top_chapter_nic, 
          category_col = "bnf_chapter",
          value_col = "nic",
          title = "Top 10 BNF Chapters by Net Ingredient Cost (NIC)", 
          ylabel = "BNF Chapter",
          xlabel = "Total NIC (£)",
          isCurrency=True,
          label_currency= "M"
) 

In [ ]:
# Top 10 BNF Chapters by prescription Items

top_chapter_items = chapter_summary.nlargest(10,"items").sort_values("items", ascending=False)

# creating a bar chart for top 10 BNF chapters by prescription items using the plotting function library
pf.plot_barh(df = top_chapter_items, 
          category_col = "bnf_chapter",
          value_col = "items",
          title = "Top 10 BNF Chapters by Prescription Items Dispensed", 
          ylabel = "BNF Chapter",
          xlabel = "Prescription Items Dispensed",
          isCurrency=False,
          label_currency= "M"
) 

In [ ]:
# Top 10 BNF chapters by total quatity dispensed 
top_chapter_total_quantity = chapter_summary.nlargest(10,"total_quantity").sort_values("total_quantity", ascending=False)

# creating a bar chart using the plotting function library
pf.plot_barh(df = top_chapter_total_quantity, 
          category_col = "bnf_chapter",
          value_col = "total_quantity",
          title = "Top 10 BNF Chapters by Total Quatity Dispensed", 
          ylabel = "BNF Chapter",
          xlabel = "Total Quantity Dispensed",
          isCurrency=False,
          label_currency= "B"
) 

In [ ]:
# saving datsets creating during this section

# saving chapter summary in BNF Chapter folder
chapter_summary.to_csv(Output_dir / "BNF Chapter/chapter_summary.csv", index=False)


# saving top 10 files 
top_chapter_nic.to_csv(Output_dir/ "BNF Chapter/top_10_chapter_nic.csv", index=False)
top_chapter_items.to_csv(Output_dir/"BNF Chapter/top_10_chapter_items.csv", index=False)
top_chapter_total_quantity.to_csv(Output_dir/ "BNF Chapter/top_10_chapter_total_quantity.csv", index=False)


# SECTION 5: Product Level Analysis

Product Level Overview

    5.1 Unique BNF names
    5.2 Filtering exception records
    5.3 aggregating prescribing measures by products
    5.4 Top 10 products by expenditure (nic)
    5.5 Creating medicine only dataset
    5.6 Top 10 medicines by NIC 
    5.7 Top 10 medicines by items
    5.8 Top 10 medicines by total quantity



In [ ]:
# checking medicines by name

df_NHS_dataset["generic_bnf_equivalent_name"].nunique() # checking quantity: 19094

df_NHS_dataset["generic_bnf_equivalent_name"].isnull().sum() # 1 null value


In [ ]:
df_NHS_dataset[df_NHS_dataset["generic_bnf_equivalent_name"].isnull()] # checking the 1 null value

IMPORTANT NOTE FOR THIS SECTION

Exception records were excluded from medicine-level summaries because they do not represent identifiable medicines. These records remained in the master dataset and accounted for only 0.04% of all observations.

In [ ]:
# filtering the dataset to include no exception records

df_analysis = df_NHS_dataset[~df_NHS_dataset["is_exception_record"]].copy()

df_analysis.reset_index(drop=True, inplace=True)
df_analysis.info()

In [ ]:
# checking unique generic names 
df_analysis["generic_bnf_equivalent_name"].nunique()  #19092

df_analysis["generic_bnf_equivalent_name"].isnull().sum() # no nulls

In [ ]:
# aggregating prescribing measure by products

products_summary = (
            df_analysis.groupby(["generic_bnf_equivalent_name", "bnf_chapter"])[["items", "total_quantity", "nic"]]
            .sum()
            .sort_values("nic", ascending=False)
            .reset_index()
)

products_summary.head(10) # 2 appliances in the top 10 products

In [ ]:
# top products by expenditure (nic)

top_10_products_nic = products_summary.nlargest(10,"nic").sort_values("nic", ascending=False)

# plotting a horizontal bar chart

pf.plot_barh(df = top_10_products_nic, 
          category_col = "generic_bnf_equivalent_name",
          value_col = "nic",
          title = "Top 10 Products by Net Ingredient Cost (NIC)", 
          ylabel = "Product (BNF Chapter)",
          xlabel = "Total NIC (£)",
          isCurrency=True,
          label_currency= "M",
          extra_label = "bnf_chapter"
        ) 


The top 10 products include medicines used in endocrine, respiratory, and cardiovascular care, alongside non-pharmaceutical products such as glucose monitoring sensors and catheters, highlighting that NHS prescribing expenditure includes both medicines and medical devices.

##### Creating a dataset for only medicine categories for efficient analysis

In [ ]:
# create a list of non-medical categories

non_medical_categories = [
    "Appliances",
    "Stoma Appliances",
    "Dressings",
    "Incontinence Appliances",
    "Preparations used in Diagnosis"
]


df_medicine = df_analysis[~df_analysis["bnf_chapter"].isin(non_medical_categories)].copy()

df_medicine["bnf_chapter"].value_counts()

In [ ]:
medicine_summary = (
    df_medicine
    .groupby(
        ["generic_bnf_equivalent_name", "bnf_chapter"]
    )[["items", "total_quantity", "nic"]]
    .sum()
    .sort_values("nic", ascending=False)
    .reset_index()
)

medicine_summary.head(10) # endocrine and respiratory medicine are dominating the top costs

In [ ]:
# top 10 medicines by NIC 

top_10_medicines_nic = medicine_summary.nlargest(10, "nic").sort_values("nic", ascending=False)

# plotting a horizontal bar chart

pf.plot_barh(df = top_10_medicines_nic, 
          category_col = "generic_bnf_equivalent_name",
          value_col = "nic",
          title = "Top 10 Medicines by Net Ingredient Cost (NIC)", 
          ylabel = "Medicine (BNF Chapter)",
          xlabel = "Total NIC (£)",
          isCurrency=True,
          label_currency= "M",
          extra_label = "bnf_chapter"
        ) 


Endocrine System products accounted for the highest Net Ingredient Cost. This was largely driven by newer antidiabetic therapies, including tirzepatide and dapagliflozin, which are substantially more expensive than many traditional medicines. These findings highlight the financial impact of modern diabetes treatments on NHS prescribing expenditure.

In [ ]:
# top 10 medicines by items

top_10_medicines_item = medicine_summary.nlargest(10, "items").sort_values("items", ascending=False)

# plotting a horizontal bar chart

pf.plot_barh(df = top_10_medicines_item, 
          category_col = "generic_bnf_equivalent_name",
          value_col = "items",
          title = "Top 10 Medicines by Prescription Items", 
          ylabel = "Medicine (BNF Chapter)",
          xlabel = "Total Item Prescription",
          isCurrency=False,
          label_currency= "M",
          extra_label = "bnf_chapter"
        ) 


Cardiovascular and Gastro-Intestinal medicines accounted for the highest numbers of prescription items, reflecting the widespread management of common chronic conditions.

In [ ]:
# top medicines by total quantity

top_10_medicines_total_quantity = medicine_summary.nlargest(10, "total_quantity").sort_values("total_quantity", ascending=False)

# plotting a horizontal bar chart

pf.plot_barh(df = top_10_medicines_total_quantity, 
          category_col = "generic_bnf_equivalent_name",
          value_col = "total_quantity",
          title = "Top 10 Medicines by Total Quantity", 
          ylabel = "Medicine (BNF Chapter)",
          xlabel = "Total Quantity Dispensed",
          isCurrency=False,
          label_currency= "M",
          extra_label = "bnf_chapter"
        ) 


In [ ]:
# CHECKING TOP 10 products prescribed items in nutrition and blood based on total quantity
df_NHS_dataset[
    df_NHS_dataset["bnf_chapter"] == "Nutrition and Blood"
].groupby("generic_bnf_equivalent_name")["total_quantity"] \
 .sum() \
 .sort_values(ascending=False) \
 .head(10)

The Nutrition and Blood chapter accounted for the highest total quantity dispensed, primarily driven by oral nutritional supplement products such as Ensure and Fortisip. Unlike prescription item counts or expenditure, total quantity is influenced by formulation type and dispensing volume, with liquid nutritional products contributing substantially to overall quantities.

In [ ]:
# saving the created datasets

# saving the analysis dataset (this dataset has no exception records) in the cleaned data folder
df_analysis.to_parquet("../data/cleaned data/nhs_prescribing_analysis.parquet", index=False)

# saving the medicine dataset (that only medicine records with no exceptions) in the cleaned data folder
df_medicine.to_parquet("../data/cleaned data/nhs_prescribing_medicine_only.parquet", index=False)

# saving medicines summary dataset in the product medicine folder
#medicine_summary = medicine_summary.reset_index(drop=True)
medicine_summary.to_parquet(Output_dir / "Product_Medicine/medicine_summary.parquet", index=False)


# saving products summary dataset
#products_summary = products_summary.reset_index(drop=True)
products_summary.to_parquet(Output_dir / "Product_Medicine/products_summary.parquet", index=False)
#products_summary.to_csv(Output_dir / "Product_Medicine/products_summary.csv", index=False)


# saving top 10 files 
top_10_products_nic.to_csv(Output_dir / "Product_Medicine/top_10_products_nic.csv", index=False)
top_10_medicines_nic.to_csv(Output_dir / "Product_Medicine/top_10_medicines_nic.csv", index=False)
top_10_medicines_item.to_csv(Output_dir / "Product_Medicine/top_10_medicines_item.csv", index=False)
top_10_medicines_total_quantity.to_csv(Output_dir / "Product_Medicine/top_10_medicines_total_quantity.csv", index=False)



### SECTION SUMMARY

Product-level analysis revealed different drivers of NHS prescribing activity depending on the measure examined. Prescription volume was dominated by products used for common chronic conditions, while Net Ingredient Cost was influenced by higher-cost therapies including endocrine treatments and respiratory combination inhalers. Total quantity was primarily driven by nutritional supplement products supplied in large volumes. The analysis also highlighted that NHS prescribing data includes both medicines and healthcare products such as monitoring devices and appliances.

# SECTION 6: Geographic Analysis

Geographical Overview 

    6.1 Regional Names
    6.2 Prescription items by Region
    6.3 Total Quantity Dispensed by Region
    6.4 Net Ingredient Cost by Region
    6.5 NIC per prescription item by region
    6.6 Total Quantity per Prescription Item by Region


In [ ]:
# regional Prescribing overview

df_analysis["region_name"].value_counts()
# how many regions are there (7)

In [ ]:
# region summary chart

region_summary = (
                    df_analysis
                    .groupby("region_name")[["items", "total_quantity", "nic"]]
                    .sum()
                    .sort_values("nic", ascending=False)
                    .reset_index()
)

region_summary

In [ ]:
# prescription items by region

region_breakdown = region_summary.sort_values("items", ascending=False)


# plotting the chart

pf.plot_barh(df = region_breakdown, 
          category_col = "region_name",
          value_col = "items",
          title = "Prescription Items by NHS Regions", 
          ylabel = "Region",
          xlabel = "Items Precribed",
          isCurrency=False,
          label_currency= "M"
        ) 


In [ ]:
# total quantity dispensed by region

region_breakdown = region_summary.sort_values("total_quantity", ascending=False)

# plotting the chart

pf.plot_barh(df = region_breakdown, 
          category_col = "region_name",
          value_col = "total_quantity",
          title = "Total Quantity Dispensed by NHS Regions", 
          ylabel = "Region",
          xlabel = "Total Quantity Dispensed",
          isCurrency=False,
          label_currency= "B"
        ) 


In [ ]:
# nic by region

region_breakdown = region_summary.sort_values("nic", ascending=False)

# plotting the chart

pf.plot_barh(df = region_breakdown, 
          category_col = "region_name",
          value_col = "nic",
          title = "Net Ingredient Cost by NHS Regions", 
          ylabel = "Region",
          xlabel = "Net Ingredient Cost",
          isCurrency=True,
          label_currency= "M"
        ) 

Regional analysis showed that Midlands and North East and Yorkshire contributed the highest overall prescribing activity across prescription items, total quantity dispensed, and Net Ingredient Cost. This pattern is consistent with these regions having larger populations and therefore greater overall prescribing demand.

In [ ]:
# Calculating NIC per prescription item by region

region_summary["nic_per_item"] = region_summary["nic"] / region_summary["items"]

region_summary = region_summary.sort_values("nic_per_item", ascending=False)

region_summary

In [ ]:
# Calculating total quantity per prescription item by region

region_summary["quantity_per_item"] = region_summary["total_quantity"] / region_summary["items"]

region_summary = region_summary.sort_values("quantity_per_item", ascending=False)

region_summary

Although Midlands and North East and Yorkshire accounted for the highest overall prescribing activity, normalised measures revealed differences in prescribing patterns between regions. South East and East of England demonstrated higher average cost per prescription item, suggesting a greater contribution from higher-cost products. In contrast, Midlands had the highest quantity dispensed per prescription item, indicating larger average dispensing volumes.

In [ ]:
# plotting cost per item

region_breakdown = region_summary.sort_values("nic_per_item", ascending=False)

# plotting the chart

pf.plot_barh(df = region_breakdown, 
          category_col = "region_name",
          value_col = "nic_per_item",
          title = "Net Ingredient Cost per Item by NHS Regions", 
          ylabel = "Region",
          xlabel = "Cost per Item",
          isCurrency=True,
          label_currency= "S"
        ) 


In [ ]:
# saving the regional dataset
#region_summary = region_summary.reset_index(drop=True)
region_summary.to_csv(Output_dir / "Geography/region_summary.csv", index=False)


# SECTION 7: ICB Analysis

ICB Overview

    7.1 ICB unique names
    7.2 Merging ICB Board Names
    7.3 Top 10 ICB by Prescription Items
    7.4 Top 10 ICB by NIC
    7.5 Top 10 ICB by Total Quantity
    7.6 NIC per Item
    7.7 Total Quantity Per Item
##### IMPORTANT NOTE: ICB-level comparisons should be interpreted with consideration of recent NHS organisational changes, which may affect geographic groupings and comparability.

In [ ]:
# how many unique ICB names are there
df_analysis["icb_name"].nunique() # 72 ICB boards

In [ ]:
# analysis by ICB board

icb_summary = (
    df_analysis
    .groupby("icb_name")[["items", "total_quantity", "nic"]]
    .sum()
    .sort_values("nic", ascending=False)

    .reset_index()
)

icb_summary

# top three ICB boards by nic are: NHS West Yorkshire, NHS Greater Manchester, and NHS North East and North Cumbria

### important note

ICB-level analysis identified multiple naming variations associated with April 2026 organisational changes. These were retained to preserve the original data structure; however, a new column was created for further analysis.  This reduced the number of unique ICB names from 72 to 48.

In [ ]:
# MERGING ICB NAME
# creating a new column for merged names
icb_summary["icb_name_clean"] = (
    icb_summary["icb_name"]
    .str.replace(r"\s*\(C.*?\)","", regex=True)
    .str.strip()
)

In [ ]:
icb_summary["icb_name_clean"].nunique()
# after merging

In [ ]:
icb_summary["icb_name_clean"].value_counts()

In [ ]:
# analysis by new cleaned ICB board column

icb_summary_2 = (
    icb_summary
    .groupby("icb_name_clean")[["items", "total_quantity", "nic"]]
    .sum()
    .sort_values("nic", ascending=False)

    .reset_index()
)

icb_summary_2.head(10)

# top three ICB boards by nic are: NHS West Yorkshire, NHS Greater Manchester, and NHS North East and North Cumbria

### visualisations

In [ ]:
# plotting top 10 ICB by Prescription Items

top_10_ICB_Items = icb_summary_2.nlargest(10, "items").sort_values("items", ascending=False)

# plotting the chart

pf.plot_barh(df = top_10_ICB_Items, 
          category_col = "icb_name_clean",
          value_col = "items",
          title = "Top 10 Boards by Prescription Items", 
          ylabel = "ICB Name",
          xlabel = "Prescription Items",
          isCurrency=False,
          label_currency= "M"
        ) 


In [ ]:
# plotting top 10 ICB by NIC

top_10_ICB_nic = icb_summary_2.nlargest(10, "nic").sort_values("nic", ascending=False)

# plotting the chart

pf.plot_barh(df = top_10_ICB_nic, 
          category_col = "icb_name_clean",
          value_col = "nic",
          title = "Top 10 Boards by Net Ingredient Costs (NIC)", 
          ylabel = "ICB Name",
          xlabel = "Net Ingredient Cost (NIC)",
          isCurrency=True,
          label_currency= "M"
        ) 


In [ ]:
# plotting top 10 ICB by quantity

top_10_ICB_quantity = icb_summary_2.nlargest(10, "nic").sort_values("total_quantity", ascending=False)

# plotting the chart

pf.plot_barh(df = top_10_ICB_quantity, 
          category_col = "icb_name_clean",
          value_col = "total_quantity",
          title = "Top 10 Boards by Total Quantity Dispensed", 
          ylabel = "ICB Name",
          xlabel = "Total Quantity Dispensed",
          isCurrency=False,
          label_currency= "B"
        ) 


Total quantity was analysed as a descriptive measure; however, quantity-based comparisons should be interpreted cautiously as values are influenced by product formulation and dispensing units, particularly liquid nutritional preparations.

In [ ]:
# checking nic per item
icb_summary_2["nic_per_item"] = icb_summary_2["nic"] / icb_summary_2["items"]

# display
icb_summary_2.sort_values("nic_per_item", ascending=False).head(10)


In [ ]:
# checking quantity per item statistics
icb_summary_2["quantity_per_item"] = icb_summary_2["total_quantity"] / icb_summary_2["items"]

# display the data
icb_summary_2.sort_values("quantity_per_item", ascending=False).head(10)


In [ ]:
# checking why NHS Herefordshire has high quantity per item
df_analysis[
    df_analysis["icb_name"] == 
    "NHS HEREFORDSHIRE AND WORCESTERSHIRE INTEGRATED CARE BOARD"
].groupby("generic_bnf_equivalent_name")["total_quantity"] \
.sum() \
.sort_values(ascending=False) \
.head(10)

In [ ]:
# saving created datasets

# saving the original ICB summary BEFORE NAME STANDARISATION
icb_summary.to_csv("../data/validated data/icb_summary_original.csv", index=False)

# saving the standardised ICB summary after resolving merger-naming differences
icb_summary_2.to_csv(Output_dir / "ICB/icb_summary_standardised.csv", index=False)


# saving top 10 ICB_summaries
top_10_ICB_Items.to_csv(Output_dir / "ICB//top_10_ICB_items.csv", index=False)
top_10_ICB_nic.to_csv(Output_dir / "ICB/top_10_ICB_nic.csv", index=False)
top_10_ICB_quantity.to_csv(Output_dir / "ICB/top_10_ICB_total_quantity.csv", index=False)


An unusually high quantity per prescription item was observed for NHS Herefordshire and Worcestershire ICB. Further investigation showed that this was primarily driven by oral nutritional supplement products, such as Nutrison and Fortisip liquid preparations, which contribute large dispensing quantities due to their formulation and measurement units.

# SECTION 8: Temporal Analysis

    8.1 Monthly summary
    8.2 Plotting NIC over time
    8.3 Plotting prescription items over time
    8.4 Product contributions to highest cost month
    8.5 Top BNF Chapter per month
    8.6 NIC percentage by BNF chapter
    8.7 Monthly NIC distribution by top selected BNF chapters

In [ ]:
# monthly summary

monthly_summary = (
    df_analysis
    .groupby("date")[["items", "total_quantity", "nic"]]
    .sum()
    .sort_values("date", ascending=True)
    .reset_index()
)

# setting change percentage for nic
monthly_summary["nic_change_pct"] = (
    monthly_summary["nic"].pct_change() * 100
)

# setting change percentage for item
monthly_summary["items_change_pct"] = (
    monthly_summary["items"].pct_change() * 100
)

monthly_summary


In [ ]:
# plotting NIC over time

# calling line plot function
pf.plot_line(
        df = monthly_summary,
        category_col = "date",
        value_col = "nic",
        title = "Monthly Net Ingredient Cost (NIC) Trend",
        ylabel= "Net Ingredient Cost (£)",
        xlabel= "DATE (month year)",
        isCurrency= True, 
        label_currency= "M",
        rotation=0)


NIC decreased by 7.7% from January to February 2026, before increasing 9.7% in March. April saw a marginal decline of 1.3%.

In [ ]:
# plotting Prescription Items over time

# calling line plot function
pf.plot_line(
        df = monthly_summary,
        category_col = "date",
        value_col = "items",
        title = "Monthly Items Dispensed Trend",
        ylabel= "Items Dispensed",
        xlabel= "DATE (month year)",
        isCurrency= False, 
        label_currency= "M",
        rotation=0)

The number of prescription items dispensed followed a similar pattern to NIC, with a decrease of 8.1% from January to February 2026, an increase of 9.4% in March, and a slight decline (2.1%) in April.

In [ ]:
# checking the highest cost month (March) to determine what products contributed towards the cost

df_analysis[df_analysis["date"] == "2026-03-01"].groupby("generic_bnf_equivalent_name") ["nic"] \
                                                .sum() \
                                                .sort_values(ascending=False) \
                                                .head(10)

# top 10 products contributing to NIC for March 2026

The highest NIC month (March 2026) was driven primarily by high-cost products including FreeStyle Libre 2 Plus Sensor and tirzepatide formulations. Respiratory therapies and cardiovascular medicines also contributed to expenditure, indicating that the increase was associated with multiple therapeutic areas rather than a single product category.

In [ ]:
#monthly BNF chapter trends
monthly_chapter = (
    df_analysis
    .groupby(["date","bnf_chapter"])["nic"]
    .sum()
    .reset_index()
)

#monthly_chapter

In [ ]:
top_chapter_month = (
    monthly_chapter.loc[
        monthly_chapter.groupby("date")["nic"].idxmax()
    ]
)

top_chapter_month

In [ ]:
monthly_chapter["nic_percentage"] = (
    monthly_chapter["nic"] /
    monthly_chapter.groupby("date")["nic"].transform("sum") * 100
)


monthly_chapter.nlargest(25,"nic_percentage")

Endocrine System remained the largest contributor to monthly NIC, accounting for approximately 21% of expenditure each month from January to April 2026. Central Nervous System, Respiratory System, and Cardiovascular System medicines were also consistent contributors, suggesting a stable prescribing cost distribution over the analysed period.

In [ ]:
# making a list of selected chapters
top_chapters = [
    "Endocrine System",
    "Central Nervous System",
    "Respiratory System",
    "Cardiovascular System",
    "Appliances"
]

# copyhing selected chapters into a new dataset
chapter_trend = monthly_chapter[
    monthly_chapter["bnf_chapter"].isin(top_chapters)
].copy()

# pivoting the data
chapter_pivot = (
    chapter_trend
    .pivot(
        index="date",
        columns="bnf_chapter",
        values="nic_percentage"
    )
    .fillna(0)
)


# converting dates to string format
chapter_pivot.index = (
    chapter_pivot.index.strftime("%b %Y")
)

# plotting the stacked bar chart 
# calling the plot_stacked_bar

pf.plot_stacked_bar(
        df = chapter_pivot,
        title = "Monthly NIC Distribution by BNF Chapter (Top Contributors)",
        xlabel = "Month",
        ylabel  = "Percentage of NIC (%)",
        leg_title = "BNF Chapter"
)


Although total NIC fluctuated between January and April 2026, the distribution of expenditure across major BNF chapters remained consistent

In [ ]:
# saving the summary data
monthly_summary.to_csv(Output_dir / "Temporal/monthly_summary.csv", index=False)
monthly_chapter.to_csv(Output_dir / "Temporal/monthly_chapter.csv", index=False)
chapter_trend.to_csv(Output_dir / "Temporal/chapter_trend.csv", index=False)


### EDA Summary — Key Findings

### Dataset Overview

- Analysed the NHS Prescription Cost Analysis (PCA) dataset containing approximately 2.25 million prescription records from January–April 2026.
- The dataset included information on prescription items, costs (NIC), quantities, products, BNF chapters, regions, and ICB organisations.
- The dataset contained both medicines and non-medicine products, including appliances, devices, nutritional products, and dressings.

### Data Quality and Preparation

- Identified and removed exception records that did not represent valid prescribing items.
- Investigated missing values and inconsistencies in key fields.
- Standardised ICB names to account for organisational changes and mergers, reducing duplicate representations of the same organisation.
- Created derived variables to support analysis:
    - monthly date fields
    - NIC per item
    - quantity per item
    - percentage contribution metrics

### Product-Level Analysis

- Prescribing expenditure was concentrated among a relatively small number of products.
- High-cost products included:
    - diabetes monitoring devices (e.g., FreeStyle Libre sensors)
    - endocrine treatments (e.g., tirzepatide)
    - respiratory inhalers
    - specialised nutritional products
- The highest-volume products were not always the highest-cost products, highlighting differences between prescribing frequency and expenditure impact
- Nutritional products dominated total quantity analysis, reflecting high-volume prescribing of liquid nutritional preparations.

### BNF Chapter Analysis

- The Endocrine System was the largest contributor to NIC across the analysis period
- Other major contributors to prescribing expenditure included:
  - Central Nervous System
  - Respiratory System
  - Cardiovascular System
  - Appliances
- Cardiovascular and Gastro-Intestinal System products were among the highest in terms of prescribing volume.
- Differences were observed between therapeutic areas driving cost versus those driving prescription volume.

### Geographic Analysis

- Prescribing expenditure and volume varied across regions.
- Midlands and North East & Yorkshire showed high total prescribing quantities, partly reflecting population size and regional demand.
- South East and East of England showed higher NIC per item, suggesting differences in prescribing mix and product costs.
- ICB-level analysis identified the impact of organisational changes, highlighting the importance of standardising ICB identifiers before comparison.

### Temporal Analysis (Jan–Apr 2026)

- Monthly NIC showed variation across the four-month period, with a decrease followed by an increase before a smaller decline.
- Prescription item volumes followed a similar pattern.
- The overall therapeutic area distribution remained relatively stable over time.
- Endocrine System consistently remained the largest contributor to monthly NIC.
- Monthly product drivers changed, with high-cost items such as sensors and specialised treatments contributing substantially to expenditure.

### Potential Areas for Further Analysis (RWE)

- Investigate generic versus proprietary medicine expenditure differences.
- Compare expenditure patterns between medicines, appliances, and devices.
- Analyse preparation classes (prep_class) to identify cost and volume drivers.
- Investigate geographic variation after accounting for prescribing volume.
- Identify products contributing disproportionately to total NHS prescribing expenditure.
